# 03: Scikit-learn bilan Linear Regression
Maqsad: tayyor kutubxona yordamida model qurish, baholash va hyperparameter'larni sozlash.
Dataset: Auto MPG (tozalangan, 392 ta mashina)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

df = sns.load_dataset("mpg").dropna()

Mashq 1: X, y va train_test_split

📖 Izoh: scikit-learn’da 2 ta qoida bor:

X: feature’lar (har doim 2 o’lchamli jadval, shuning uchun [["weight"]] deb ikki qavs bilan yoziladi)
y: label (1 o’lchamli ustun)

Ma’lumotni ikkiga bo’lamiz:

Train (80%): model shu qismdan o’rganadi
Test (20%): model hech qachon ko’rmagan ma’lumot. Unda modelni imtihon qilamiz.

📚 Analogiya: o’quvchiga imtihonda darslikdagi masalalarni bersangiz, u javoblarni yodlab olgan bo’lishi mumkin. Haqiqiy bilimni tekshirish uchun yangi masalalar kerak.

random_state=42 har safar bir xil bo’linish berishi uchun qo’yiladi. Shunda sizning natijangiz menikiga mos chiqadi.

In [2]:
X = df[["weight"]]
y = df["mpg"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(313, 1) (79, 1)


Model yaratish va o’qitish

📖 Izoh: scikit-learn’dagi har bir model bir xil 3 qadam bilan ishlaydi:

model = ...() → modelni yaratish
model.fit(X_train, y_train) → o’qitish (w va b ni topish)
model.predict(X) → bashorat qilish

01-notebookda yozgan butun train funksiyamiz endi bitta fit() buyrug’iga aylandi!

In [3]:
model = LinearRegression()
model.fit(X_train, y_train)

print("w (coef_):", model.coef_)
print("b (intercept_):", model.intercept_)

# 4000 funtlik mashina uchun bashorat
print(model.predict(pd.DataFrame({"weight": [4000]})))

w (coef_): [-0.00790361]
b (intercept_): 47.20052642755209
[15.58608489]


Mashq 3: Modelni baholash

📖 Izoh: Modelni test ma’lumotida tekshiramiz. MAE, MSE, RMSE’ni bilasiz, endi yangi metrika qo’shiladi:

R² (0 dan 1 gacha) modelning mpg’dagi o’zgaruvchanlikni necha foizini tushuntira olishini ko’rsatadi:

1.0 = mukammal
0.0 = model shunchaki o’rtacha qiymatni aytgandek, ya’ni foydasiz
manfiy = o’rtachani aytishdan ham yomonroq

Har safar qayta ishlatish uchun kichik funksiya yozamiz:

In [4]:
def evaluate(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)
    print(f"MAE={mae:.2f}  MSE={mse:.2f}  RMSE={rmse:.2f}  R²={r2:.3f}")

y_pred = model.predict(X_test)
evaluate(y_test, y_pred)

MAE=3.46  MSE=17.69  RMSE=4.21  R²=0.653


Mashq 4: Bir nechta feature bilan model

📖 Izoh: 02-notebookdagi korrelyatsiya jadvalidan eng foydali feature’larni olamiz. Kod deyarli o’zgarmaydi, faqat X’ga ko’proq ustun qo’shiladi. Endi model 5 ta weight topadi, har bir feature uchun bittadan:

$$y' = b + w_1x_1 + w_2x_2 + w_3x_3 + w_4x_4 + w_5x_5$$

In [5]:
features = ["weight", "horsepower", "displacement", "cylinders", "model_year"]
X = df[features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model2 = LinearRegression()
model2.fit(X_train, y_train)

print(pd.Series(model2.coef_, index=features).round(4))
evaluate(y_test, model2.predict(X_test))

weight         -0.0064
horsepower     -0.0078
displacement    0.0005
cylinders      -0.1204
model_year      0.7578
dtype: float64
MAE=2.50  MSE=10.56  RMSE=3.25  R²=0.793


Mashq 5: SGDRegressor va hyperparameter’lar

📖 Izoh: LinearRegression gradient descent’ni umuman ishlatmaydi. U javobni bitta matematik formula bilan darhol topadi. SGDRegressor esa aynan biz o’rgangan gradient descent bilan ishlaydi va hyperparameter’larni qo’lda sozlash imkonini beradi:

Parametr	Ma’nosi
eta0	Learning rate
max_iter	Epochs (maksimal soni)
learning_rate="constant"	Learning rate’ni o’zgarmas qilib qo’yish

⚠️ SGD feature’lar bir xil masshtabda bo’lishini talab qiladi, aks holda portlab ketadi. Shuning uchun StandardScalerni ishlatamiz. Uni batafsil 04-notebookda o’tamiz, hozircha shunchaki ishlatib turing.

In [6]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

for lr in [0.0001, 0.01, 0.5]:
    sgd = SGDRegressor(learning_rate="constant", eta0=lr,
                       max_iter=100, random_state=42)
    sgd.fit(X_train_s, y_train)
    print(f"learning rate = {lr}")
    evaluate(y_test, sgd.predict(X_test_s))

learning rate = 0.0001
MAE=2.73  MSE=12.29  RMSE=3.51  R²=0.759
learning rate = 0.01
MAE=2.41  MSE=10.04  RMSE=3.17  R²=0.803
learning rate = 0.5
MAE=726478310284.54  MSE=736281447388920439373824.00  RMSE=858068439804.73  R²=-14425412106455818960896.000


/Users/asi/Desktop/ml-learn/ML-Regression-practice/venv/lib/python3.13/site-packages/sklearn/linear_model/_stochastic_gradient.py:1620: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
